In [ ]:
from typing_extensions import TypedDict
from pydantic import BaseModel, Field
from typing import Literal
from langchain.chat_models import init_chat_model
from dotenv import load_dotenv
# 이 워크플로는 한 모델이 응답을 만들고(생성자), 다른 모듈이 그 응답을 평가해 피드백을 제공함. (평가자)
# 이 피드백이 '부족함'으로 판정되면 다시 생성 단계로 들어가 개선 루프를 반복함. 명확한 평가 기준이 있고, 반복 개선이 실제 품질 향상으로 이어질때 특히 효과적임.
# 글쓰기, 보고서 품질 향상, 코드 리뷰, RAG 답변 정확도 검증 등에 유용함.

# 평가자-개선자(Evaluetor-Optimizer) 워크플로는 평가 기준이 명확하고, 사람이 피드백하면 개선되는 과정이 LLM에도 적용될 수 있으며, 반복을 통해 품질 향상을 측정,확인할 수 있을때 효과적임.
# 다음 예제의 목적은 한번 생성하고, 끝내는 것이 아니라 '생성 -> 평가 -> 개선' 과정을 반복해 더 나은 결과를 만들어낸 것임.

# 01. 상태 정의.
###########################################################################
# 이를 위해 먼저 워크 플로우 전반에서 주고받을 state 와 평가자가 사용할 구조회된 출력 스키마를 정의함.

# state 는 그래프 전체에서 공유되는 전역 상태로, 현재 농담(Joke), 주제(topic), 평가자가 제공한 피드백(Feedback), 그리고 농담이 재미있는지 여부(funny or not)를 포함함.
# 이 상태는 각 노드가 실행될 때 입력으로 전달되며, 노드의 실행 결과가 다시 상태에 반영함.

# 그래프 상태.
class State(TypedDict):
    joke: str # 현재 농담
    topic: str # 주제
    feedback: str # 평가자 피드백(개선 시 참고)
    funny_or_not: str # 평가 등급("funny", "not funny")

In [ ]:
# 02. 구조회된 출력 스키마
###########################################################################

# 평가자 노드에서는 자유로운 텍스트가 아니라 일정한 형식의 결과가 필요하므로 구조화된 출력 스키마인 Feedback을 정의함.
# 이 스키마는 농담이 재미있는지 여부를 나타내는 등급 과 재미없을 경우 어떻게 개선하면 좋을 지에 대한 구체적인 피드백을 함께 담도록 설계돼 있음.
# 이후 llm.with_structured_output(Feedback) 을 통해 평가자 역할의 LLM이 항상 이 구조를 따르는 결과를 반환하도록 설정함.

# 평가에 사용할 구조화 출력 스키마
class Feedback(BaseModel):
    # Literal은 정적 타입 검사기(Pyright, MyPy)에게 이 변수나 매개변수 특정 값 그자체만 가질수 있다라고 제한할 때 사용하는 타입 힌트임
    grade: Literal["funny", "not funny"] = Field(description="농담이 재미있는지(funny) 아닌지 (not funny) 판단하세요.")
    feedback = Field(discription="재미없다면 어떻게 개선할지 구체적인 피드백을 작성하세요.")

load_dotenv()

llm = init_chat_model("openai:gpt-4.1")

# 평가자 LLM : 구조화된 출력으로 등급과 피드백을 반환.
evaluator = llm.with_structured_output(Feedback)